# Edge-Enhanced GraphSAGE — Train All Stages on Colab

Trains Stages 1 / 2 / 3a / 3 with the **v2 node features (12 dims)** and the new pattern-classifier-aware pipeline.

**Prerequisites — upload once to your Drive:**
- `/MyDrive/R26-IT-121-data/features.parquet` (~470 MB, output of `scripts/prepare_features.py`)

**Runtime:** GPU (T4 free tier is enough)  ·  Expected wall-clock: ~25 min for all four stages.

Outputs saved back to Drive at the end:
- `paysim_graph.pt`  (282 MB, the v2 graph)
- `checkpoints/stage{1,2,3a,3}.pt`
- `reports/stage{1,2,3a,3}_metrics.json`

## 1.  Clone repo, install deps, mount Drive

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/R26-IT-121/GraphSage')
BRANCH = 'deepsentinel-GraphSage'
if not REPO_DIR.exists():
    !git clone -q -b {BRANCH} https://github.com/LEXES7/R26-IT-121.git /content/R26-IT-121
%cd {REPO_DIR}
!git fetch -q --all && git checkout -q {BRANCH} && git pull -q

# Sanity-check we're on v2 code (must print 12 names ending in first_step_norm)
!grep -A 14 'NODE_FEATURE_NAMES' src/graphsage/data/graph_builder.py | head -15

# Install: torch already present on Colab; we only need PyG + project install.
!pip install -q torch-geometric pyarrow
!pip install -q -e .
sys.path.insert(0, str(REPO_DIR / 'src'))

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_DATA = Path('/content/drive/MyDrive/R26-IT-121-data')
assert DRIVE_DATA.exists(), f'Create {DRIVE_DATA} and upload features.parquet'

## 2.  Stage local data dirs + pull features.parquet from Drive

In [ ]:
(REPO_DIR / 'data' / 'processed').mkdir(parents=True, exist_ok=True)
(REPO_DIR / 'data' / 'graph').mkdir(parents=True, exist_ok=True)
(REPO_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(REPO_DIR / 'reports').mkdir(parents=True, exist_ok=True)

FEATURES_SRC = DRIVE_DATA / 'features.parquet'
FEATURES_DST = REPO_DIR / 'data' / 'processed' / 'features.parquet'
if not FEATURES_DST.exists():
    print('Copying features.parquet from Drive (≈30s)...')
    !cp "{FEATURES_SRC}" "{FEATURES_DST}"
print(f'features.parquet → {FEATURES_DST.stat().st_size // 1024**2} MB')

## 3.  Build the v2 graph (12 node features) and apply time splits

In [ ]:
!python scripts/build_graph.py
!python scripts/make_splits.py

## 4.  Train all four stages
Stage 1 → vanilla SAGE baseline · Stage 2 → + Edge-MLP · Stage 3a → + Focal Loss · Stage 3 → + Imbalance Sampler

In [ ]:
!python scripts/train_baseline.py

In [ ]:
!python scripts/train_edge_mlp.py

In [ ]:
!python scripts/train_focal.py

In [ ]:
!python scripts/train_full.py

## 5.  Threshold-tune all four stages on the validation set

In [ ]:
!python scripts/eval_with_tuned_threshold.py

## 6.  Compare to the v1 (5-feature) baseline

In [ ]:
import json
from pathlib import Path

rows = []
for stage in ['stage1', 'stage2', 'stage3a', 'stage3']:
    p = Path('reports') / f'{stage}_metrics.json'
    if p.exists():
        m = json.loads(p.read_text())
        tuned = m.get('test_metrics_tuned') or m.get('final_test_metrics') or {}
        rows.append({
            'stage': stage,
            'F1': tuned.get('f1'),
            'AUROC': tuned.get('auroc'),
            'Recall': tuned.get('recall'),
            'Precision': tuned.get('precision'),
        })

print(f"{'stage':<10}{'F1':>10}{'AUROC':>10}{'Recall':>10}{'Precision':>12}")
print('-' * 52)
for r in rows:
    print(f"{r['stage']:<10}{r['F1']:>10.4f}{r['AUROC']:>10.4f}{r['Recall']:>10.4f}{r['Precision']:>12.4f}")

print('\nv1 (5-feature) reference for comparison:')
print('  stage3a    0.5387    0.9497    0.5663      0.5132  ← previous best')

## 7.  Save graph + checkpoints + metrics back to Drive

In [ ]:
OUT = DRIVE_DATA / 'v2_run'
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'checkpoints').mkdir(exist_ok=True)
(OUT / 'reports').mkdir(exist_ok=True)

!cp data/graph/paysim_graph.pt "{OUT}/paysim_graph.pt"
!cp data/graph/split_metadata.json "{OUT}/split_metadata.json"
!cp data/graph/graph_metadata.json "{OUT}/graph_metadata.json"
!cp checkpoints/*.pt "{OUT}/checkpoints/"
!cp reports/stage*_metrics.json "{OUT}/reports/"
!cp reports/ablation_tuned.json "{OUT}/reports/" 2>/dev/null || true

print(f'Outputs saved to {OUT}. Pull these down with rsync/Drive when training is done.')